# Kokoro-82M 試聴テスト — 英会話アプリ TTS 採否判定

設計書 §09「フェーズ1.5：TTS のオンデバイス化」の **検証ステップ1** に対応するノートブック。

## 目的

**「Kokoro の音声を、英語学習者にとっての発音の手本として許容できるか」** — この一点だけを判定する。
iOS への組み込みコストやレイテンシは、ここで合格してから考える。ここで落ちたら以降の作業はすべて不要。

## 判定の考え方

一般的な TTS 評価とは基準が違う。**「聞き取りやすいか」ではなく「真似してよい発音か」** で判断する。
自然さがやや劣るのは許容できるが、以下は許容できない。

- 母音・子音の音価が英語としてずれている
- 単語の強勢位置が間違っている
- 疑問文が上昇調にならない
- 意味の切れ目と無関係な場所で区切る

## 実行方法

上から順に実行するだけ。GPU は不要（CPU ランタイムで動く）。所要 5〜10 分。

> **注意** 最後のセルで全音声を ZIP でダウンロードできる。**クラウド TTS で同じ文を生成して聴き比べる**のが本来の A/B テスト。片方だけ聴くと基準が甘くなる。

---
## 0. セットアップ

Colab の Python は **3.13**。`kokoro` の依存にある `spacy-curated-transformers` の最新版が
`spacy==4.0.0.dev` を要求し、その**ソースビルドが失敗する**（これが今回のエラーの原因）。

下のセルは2つの経路を順に試し、成功した方を自動で使う。

| 経路 | G2P | 備考 |
|---|---|---|
| **1. `kokoro`（本家）** | misaki | `spacy<4` に固定して dev 版の混入を防ぐ。**iOS 実装（MisakiSwift）と同じ G2P** |
| **2. `kokoro-onnx`** | espeak-ng | 依存が少なく Python 3.13 でも通りやすい。音響モデルの重みは同一 |

> **経路2 になった場合の注意**
>
> 音素化が espeak-ng になるため、**セクション2 の「同形異音語(heteronym)」テストだけは
> 実際の iOS 実装より不利な結果**が出る。iOS では misaki 相当の `MisakiSwift` が
> 品詞解析で read / wind を判別するため、この項目は割り引いて評価すること。
>
> それ以外の項目（音価・強勢・イントネーション・区切り・速度変更）は
> **音響モデルが同一なので結果も同じ**になる。判定への影響はない。


In [ ]:
import subprocess, sys, os, platform

print("Python", sys.version.split()[0], "/", platform.machine())


def pip(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-2500:])
    return r.returncode == 0


# espeak-ng: 経路2 の音素化に必須 / 経路1 では OOV フォールバックに使われる
subprocess.run("apt-get -qq install -y espeak-ng", shell=True, capture_output=True)

BACKEND = None

# ── 経路1: 本家 kokoro ─────────────────────────────────────────
# spacy を v3 系に固定し、spacy 4.0.0.dev の sdist ビルドを回避する
print("\n[1/2] kokoro (misaki G2P) を試行 ...")
if pip("soundfile", "spacy<4", "spacy-curated-transformers<2", "kokoro"):
    try:
        from kokoro import KPipeline          # noqa: F401
        BACKEND = "kokoro"
    except Exception as e:
        print("  import 失敗:", type(e).__name__, e)

# ── 経路2: kokoro-onnx ────────────────────────────────────────
if BACKEND is None:
    print("\n[2/2] kokoro-onnx にフォールバック ...")
    if pip("kokoro-onnx", "soundfile"):
        BASE = ("https://github.com/thewh1teagle/kokoro-onnx"
                "/releases/download/model-files-v1.0")
        for f in ["kokoro-v1.0.onnx", "voices-v1.0.bin"]:
            if not os.path.exists(f):
                print("  downloading", f)
                subprocess.run(["wget", "-q", f"{BASE}/{f}"])
        try:
            from kokoro_onnx import Kokoro     # noqa: F401
            BACKEND = "onnx"
        except Exception as e:
            print("  import 失敗:", type(e).__name__, e)

print("\n" + "=" * 46)
if BACKEND:
    print(f"OK   backend = {BACKEND}")
    if BACKEND == "onnx":
        print("※ G2P は espeak-ng。heteronym テストのみ割り引いて評価すること")
else:
    print("両経路とも失敗。上のログを共有してください")
print("=" * 46)


In [ ]:
import time, os, zipfile
import numpy as np
import soundfile as sf
from IPython.display import display, Audio, Markdown

assert BACKEND, "セットアップセルが成功していません"

SR = 24000          # Kokoro の出力サンプルレート
OUT = "samples"
os.makedirs(OUT, exist_ok=True)

# ── バックエンドの違いを吸収する（設計書と同じ「ポート」の考え方）──
if BACKEND == "kokoro":
    from kokoro import KPipeline
    _PIPE = {"a": KPipeline(lang_code="a"),   # American English
             "b": KPipeline(lang_code="b")}   # British English

    def _generate(text, voice, speed):
        p = _PIPE["b" if voice[0] == "b" else "a"]
        chunks = [a for _, _, a in p(text, voice=voice, speed=speed)]
        return np.concatenate(chunks) if chunks else np.zeros(1)

else:  # onnx
    from kokoro_onnx import Kokoro
    _K = Kokoro("kokoro-v1.0.onnx", "voices-v1.0.bin")

    def _generate(text, voice, speed):
        samples, _ = _K.create(
            text, voice=voice, speed=speed,
            lang="en-gb" if voice[0] == "b" else "en-us")
        return np.asarray(samples)


def synth(text, voice="af_heart", speed=1.0):
    """合成して (audio, 生成秒数, 音声長さ秒) を返す"""
    t0 = time.time()
    audio = np.asarray(_generate(text, voice, speed), dtype=np.float32)
    elapsed = time.time() - t0
    return audio, elapsed, len(audio) / SR


def play(label, text, voice="af_heart", speed=1.0, save_as=None):
    audio, elapsed, dur = synth(text, voice=voice, speed=speed)
    rtf = elapsed / dur if dur else float("nan")
    display(Markdown(
        f"**{label}** &nbsp;·&nbsp; `{voice}` &nbsp;·&nbsp; speed={speed} "
        f"&nbsp;·&nbsp; <span style='opacity:.6'>{dur:.1f}s / RTF {rtf:.3f}</span><br>"
        f"<span style='opacity:.7'>{text}</span>"
    ))
    display(Audio(audio, rate=SR))
    if save_as:
        sf.write(f"{OUT}/{save_as}.wav", audio, SR)
    return audio


print(f"ready  (backend = {BACKEND})")


---
## 1. ボイス選定 — 同じ文を複数の声で

まず「どの声を採用するか」を決める。**聞き心地ではなく、発音の明瞭さと強勢の正確さ**で選ぶこと。
`af_heart` は品質評価が最も高いとされる標準ボイス。

In [ ]:
VOICE_TEST = (
    "So, how was your weekend? I heard you went hiking with your family. "
    "Did you manage to reach the summit, or did the weather turn bad?"
)

VOICES = [
    ("af_heart",   "US 女性 / 標準・最高評価"),
    ("af_bella",   "US 女性 / 落ち着いた声"),
    ("af_nicole",  "US 女性 / 柔らかめ"),
    ("am_michael", "US 男性 / 標準"),
    ("am_fenrir",  "US 男性 / 低め"),
    ("bf_emma",    "UK 女性"),
    ("bm_george",  "UK 男性"),
]

for v, desc in VOICES:
    play(desc, VOICE_TEST, voice=v, save_as=f"01_voice_{v}")

---
## 2. プロソディ耐久テスト

**ここが本番。** TTS が破綻しやすいパターンを意図的に並べてある。
上のセルで選んだボイスを `PICK` に設定して実行する。

1つでも致命的に崩れるものがあれば、その時点で不採用を検討してよい。

In [ ]:
PICK = "af_heart"   # ← セクション1 で気に入った声に変える

STRESS_TESTS = [
    ("疑問文の上昇調",
     "Are you coming with us? Really? You didn't tell me that."),

    ("WH疑問文（下降調）との対比",
     "Where did you go last night? And who were you with?"),

    ("縮約形の連続",
     "I'd've told you if I'd known, but you weren't answering, "
     "and I couldn't've waited any longer."),

    ("対比強勢（意味が強勢に依存）",
     "I didn't say he stole the money. I said he BORROWED it."),

    ("同形異音語 heteronym",
     "I read that book last year. Now I read it again. "
     "The wind was too strong to wind the rope."),

    ("数字・時刻・日付",
     "The meeting is at 3:45 PM on March 3rd, 2026. "
     "It costs $1,250.50, which is about 15% more than last year."),

    ("略語・頭字語",
     "The CEO of NASA will discuss the AI and IoT roadmap in the FAQ section."),

    ("長文での間の取り方",
     "When you're learning a language, the hardest part isn't grammar or "
     "vocabulary — it's getting comfortable with making mistakes in front of "
     "other people, which takes far more practice than most textbooks admit."),

    ("辞書外語（G2P フォールバック）",
     "Let's grab some ramen in Shibuya, then visit Kiyomizu-dera in Kyoto."),

    ("最小対立ペア（発音の手本として致命的）",
     "She sells sheep on the ship. I need a full bowl of rice. "
     "Think about that thing. He works at the world's largest firm."),
]

for i, (label, text) in enumerate(STRESS_TESTS, 1):
    play(f"{i:02d}. {label}", text, voice=PICK, save_as=f"02_stress_{i:02d}")

---
## 3. 速度調整 — 学習者向けモード

英会話アプリでは**レベルに応じて話速を落とす**ことが多い。
速度を変えたときに音質が破綻しないか、不自然に間延びしないかを確認する。

In [ ]:
SPEED_TEXT = (
    "That's an interesting point. Could you explain what you mean by that? "
    "I'm not sure I follow."
)

for sp in [0.7, 0.85, 1.0, 1.15]:
    play(f"speed = {sp}", SPEED_TEXT, voice=PICK, speed=sp,
         save_as=f"03_speed_{str(sp).replace('.', '_')}")

---
## 4. 会話ターンの実地シミュレーション

実際のアプリで AI が返しそうな長さ・語り口の文を並べる。
**1〜2文の短い応答**が実際の使われ方なので、ここでの自然さが体験に直結する。

In [ ]:
TURNS = [
    "Hey! Good to see you again. What have you been up to this week?",
    "Oh nice, that sounds fun. How long have you been doing that?",
    "Hmm, I see what you mean. Actually, a more natural way to say that "
    "would be \"I've been working on it for three months.\"",
    "Don't worry about it — that's a really common mistake. Want to try again?",
    "Perfect, that sounded much more natural. Let's move on to something else.",
]

for i, t in enumerate(TURNS, 1):
    play(f"turn {i}", t, voice=PICK, save_as=f"04_turn_{i}")

---
## 5. 生成速度の参考計測

> **注意** Colab は x86 CPU（または T4 GPU）で、**iPhone の Neural Engine とは全く別物**。
> ここで得られる RTF は iPhone 16e の実性能を示さない。
> 「モデルが病的に遅くないか」の確認と、文長による傾向を見るためだけに使う。
> **実機での実測は §09 検証ステップ3 で別途必要。**

In [ ]:
LENGTHS = {
    "短文 (1文)": "That sounds great.",
    "中文 (2文)": "That sounds great. How long have you been planning it?",
    "長文 (4文)": " ".join(TURNS[:4]),
}

synth("warm up.", voice=PICK)   # 初回は重いので捨てる

print(f"{'ケース':<14}{'音声長':>9}{'生成時間':>11}{'RTF':>9}")
print("-" * 45)
for label, text in LENGTHS.items():
    _, elapsed, dur = synth(text, voice=PICK)
    print(f"{label:<14}{dur:>8.2f}s{elapsed:>10.2f}s{elapsed/dur:>9.3f}")

print("\n※ RTF < 1.0 ならリアルタイムより速い。文単位の逐次合成で体感レイテンシは")
print("   最初の1文の生成時間で決まるため、短文の値が最も重要。")

---
## 6. 音声を書き出して A/B テストへ

**このステップを飛ばさないこと。** Kokoro だけを聴いていると耳が慣れて基準が甘くなる。
同じ文をクラウド TTS でも生成し、**どちらが Kokoro か分からない状態で**聴き比べるのが本来の A/B。

In [ ]:
zip_path = "kokoro_samples.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT)):
        z.write(os.path.join(OUT, f), f)

print(f"{len(os.listdir(OUT))} files -> {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("（Colab 以外では手動で取得してください）", e)

In [ ]:
# A/B 用に、比較対象へ投げるテキストをそのまま出力する
print("=== セクション2 プロソディ耐久テスト（クラウド TTS でも同じ文を生成すること）===\n")
for i, (label, text) in enumerate(STRESS_TESTS, 1):
    print(f"[{i:02d}] {label}")
    print(text.replace("\n", " "))
    print()

---
## 7. 判定チェックリスト

以下をすべて聴き終えてから記入する。**1つでも「不可」があれば採用を見送る。**

| # | 判定項目 | 基準 | 可 / 不可 |
|---|---|---|---|
| 1 | 母音・子音の音価 | 最小対立ペア（sheep/ship, bowl/ball, think/sink）が明確に区別できる | |
| 2 | 単語の強勢位置 | 誤った音節に強勢が置かれていない | |
| 3 | 疑問文のイントネーション | Yes/No 疑問文が上昇調、WH 疑問文が下降調になっている | |
| 4 | 区切りの位置 | 意味の切れ目で区切っている（語句の途中で切れない） | |
| 5 | 縮約形 | I'd've / couldn't've が破綻しない | |
| 6 | 数字・略語 | 3:45 PM、$1,250.50、CEO が正しく読まれる | |
| 7 | 速度変更時の品質 | speed=0.7 でも不自然に間延びしない | |
| 8 | 短い応答の自然さ | 1〜2文の相槌が機械的に聞こえない | |
| 9 | 長時間の聴取疲労 | 10分間聴き続けても不快でない | |
| 10 | A/B 比較 | クラウド TTS と並べても「手本」として許容できる | |

### 判定が「可」だった場合の次のステップ

設計書 §09 の検証ステップ3 へ進む。

- **G2P の問題は解決済み** — `MisakiSwift`（Apache-2.0）が Swift 移植を提供しており、`espeak-ng`（GPLv3）は不要
- 有力候補は `soniqo/speech-swift` の `KokoroTTS`（Core ML / ANE 実行・INT8 量子化あり・約80〜170MB）
- 次にやること：**iPhone 16e 実機で RTF とメモリ、初回ウォームアップ時間を実測**

### 「不可」だった場合

クラウド TTS のまま進める。設計書 §09 は保留のまま残し、フェーズ2 に進む。
この判定に至った理由（どの項目が不可だったか）を §14 未決事項に記録しておくと、
将来モデルが更新されたときに再評価の基準として使える。